# 📕 Notebook 3 — Report tabellare per la dirigenza del Liceo Virgilio

Nei due notebook precedenti abbiamo lavorato con NumPy: un'analisi diagnostica sulla 3A (📘) e una visione multidimensionale sull'intero terzo anno (📗). I risultati sono stati matrici e tensori — strumenti potenti per l'analista, ma non per chi quei risultati deve poi **leggerli, condividerli e usarli operativamente**.

La dirigenza, dopo aver visto le matrici aggregate del Notebook 2, ha posto la richiesta che chiude il giro:

> *"Abbiamo bisogno di un formato più leggibile per i nostri incontri settimanali. Possiamo avere tutti questi dati in tabelle strutturate invece che in array NumPy?"*

In questo notebook **cambiamo strumento**: passiamo da **NumPy a Pandas**, la libreria che porta nel mondo Python il concetto di tabella etichettata (Series e DataFrame). E cambiamo anche **registro**: dall'analisi al deliverable, dal codice esplorativo al report consegnabile.

---

## 📂 Il caso d'uso: il programma di tutoraggio

Il Liceo Virgilio ha attivato un programma di tutoraggio per gli studenti del terzo anno con difficoltà specifiche. Sono coinvolte **due fonti di dati** che andrebbero confrontate:

| Fonte | Origine | Natura del dato in Python |
|-------|---------|---------------------------|
| **Segreteria** | Elenco compilato manualmente dal personale di segreteria | Lista Python (`list`) |
| **Sistema informatico** | Export automatico di studenti segnalati dal sistema | Array NumPy (`np.ndarray`) |

Le due fonti contengono **ID studenti** (numeri interi compresi tra 0 e 20). Le due liste possono coincidere parzialmente — ed è proprio dalla loro **riconciliazione** che emergono le informazioni più utili: chi è tracciato da entrambe le fonti? E chi invece compare solo da una parte?

---

## 🧭 Cosa contiene questo notebook

1. **Creazione delle due Serie** (segreteria e sistema)
2. **Filtro modulare** sulla serie del sistema (multipli di 3)
3. **Costruzione del DataFrame combinato** con statistiche descrittive
4. **Riconciliazione delle due fonti** (intersezione e differenza)
5. **Reshape di una serie lunga in DataFrame**: 35 valori in tabella 7 × 5
6. **Extra: report finale per la dirigenza** — DataFrame dei punteggi degli studenti tutorati nelle quattro materie classiche, con media e deviazione standard per materia

Il notebook chiude con la sintesi finale del progetto: cosa abbiamo costruito, cosa la dirigenza si porta a casa, cosa avrebbe senso fare in un ipotetico passo successivo.

In [1]:
import numpy as np
import pandas as pd

# Riproducibilità: lo stesso seed garantisce un dataset deterministico.
# Cambiando questo valore si ottengono ID diversi ma con la stessa struttura.
np.random.seed(7)

# Fonte 1 — Segreteria: elenco manuale compilato dal personale.
# Naturalmente espresso come lista Python e poi convertito in Series.
lista_segreteria = np.random.randint(0, 21, 10).tolist()
serie_segreteria = pd.Series(lista_segreteria, name="segreteria")

# Fonte 2 — Sistema informatico: export automatico, restituito come array NumPy.
array_sistema = np.random.randint(0, 21, 10)
serie_sistema = pd.Series(array_sistema, name="sistema")

# Visualizzazione delle due Serie
print("Programma di tutoraggio — fonti dati a confronto")
print("-" * 60)
print("Fonte 1 — Segreteria (compilazione manuale):")
print(serie_segreteria)
print()
print("Fonte 2 — Sistema informatico (export automatico):")
print(serie_sistema)

Programma di tutoraggio — fonti dati a confronto
------------------------------------------------------------
Fonte 1 — Segreteria (compilazione manuale):
0    15
1     4
2     3
3    19
4     7
5    14
6     8
7    14
8    10
9     8
Name: segreteria, dtype: int64

Fonte 2 — Sistema informatico (export automatico):
0     7
1     6
2     4
3    16
4     7
5    12
6     0
7    11
8     6
9    19
Name: sistema, dtype: int32


## 2️⃣ Task 2 — Filtro modulare sulla serie del sistema

Questo task ha **carattere prevalentemente tecnico**. Come anticipato nel Notebook 0, alcuni passaggi della traccia d'esame dimostrano una specifica tecnica di manipolazione senza rispondere a una domanda di business diretta del Liceo. Il filtro modulare — *trovare le posizioni dei multipli di 3 in una Series* — è uno di questi.

### La tecnica e a cosa serve in generale

Il **filtro modulare** in Pandas combina due elementi:
- una **maschera booleana** costruita con un'operazione vettoriale (`serie % 3 == 0`)
- l'attributo `.index` della Series filtrata, che restituisce le **etichette** (qui: posizioni numeriche) degli elementi che soddisfano la maschera

È un pattern molto generale che torna utile ogni volta che si vogliono identificare elementi periodici, ricorrenti o appartenenti a una qualche regola modulare. Tipici casi d'uso reali: estrarre le righe pari di un dataset, isolare elementi a intervalli regolari, identificare ID che seguono pattern algoritmici. Qui il caso d'uso è dimostrativo — applichiamo la tecnica alla serie del sistema informatico, senza una particolare interpretazione narrativa.

In [2]:
def trovare_posizioni_multiplo_3(serie_B):
    """
    Trova gli indici dei valori multipli di 3 in una pandas Series.

    Args:
        serie_B (pandas.Series): Series contenente valori numerici.

    Returns:
        pandas.Index: Gli indici dei valori che sono multipli di 3.
    """
    # Maschera booleana: True dove l'elemento è divisibile per 3.
    # Selezioniamo poi solo gli indici di quegli elementi.
    indici = serie_B[serie_B % 3 == 0].index

    # Stampa formattata con intestazione coerente con gli altri task
    print("Posizioni dei multipli di 3 nella serie del sistema")
    print("-" * 60)
    print(f"Indici trovati: {list(indici)}")
    print(f"Valori corrispondenti: {list(serie_B[indici])}")
    print(f"Numero di elementi multipli di 3: {len(indici)}")

    return indici


# Applicazione alla serie del sistema informatico
posizioni_multipli_3 = trovare_posizioni_multiplo_3(serie_sistema)

Posizioni dei multipli di 3 nella serie del sistema
------------------------------------------------------------
Indici trovati: [1, 5, 6, 8]
Valori corrispondenti: [6, 12, 0, 6]
Numero di elementi multipli di 3: 4


## 3️⃣ Task 3 — Combinazione delle due fonti in un DataFrame

Affianchiamo le due Series come colonne di un unico DataFrame, con un indice personalizzato (`Studente_01, ..., Studente_10`) e un nome assegnato all'indice. È utile premettere una precisazione: il DataFrame combinato affianca i due elenchi **riga per riga in ordine di posizione**, non per corrispondenza semantica. La riga *Studente_01* non rappresenta "lo stesso studente nelle due fonti": le due Series sono elenchi indipendenti, e il DataFrame le mostra come due colonne parallele. La riconciliazione vera — chi compare in entrambe le liste, chi solo in una — la affronteremo nei Task 4 e 5.

Aggiungiamo poi `describe()`, il metodo che produce in una sola chiamata il sommario statistico standard per ogni colonna numerica (count, media, deviazione standard, min, quartili, max). È spesso il primo sguardo che si dà a un dataset.

In [3]:
# Combinazione delle due Serie in un DataFrame:
# segreteria e sistema diventano due colonne affiancate.
df_tutorato = pd.DataFrame({
    "segreteria": serie_segreteria,
    "sistema": serie_sistema
})

# Indice personalizzato narrativo: Studente_01, Studente_02, ..., Studente_10
# Lo zero-padding (01, 02) garantisce ordinamento alfabetico corretto in caso
# di stampa o esportazione.
indici_personalizzati = [f"Studente_{i+1:02d}" for i in range(len(df_tutorato))]
df_tutorato.index = indici_personalizzati
df_tutorato.index.name = "ID Tutorato"

# Visualizzazione del DataFrame combinato
print("DataFrame combinato — fonti dati affiancate")
print("-" * 60)
print(df_tutorato)

# Statistiche descrittive per colonna
print()
print("Statistiche descrittive per colonna")
print("-" * 60)
print(df_tutorato.describe())

DataFrame combinato — fonti dati affiancate
------------------------------------------------------------
             segreteria  sistema
ID Tutorato                     
Studente_01          15        7
Studente_02           4        6
Studente_03           3        4
Studente_04          19       16
Studente_05           7        7
Studente_06          14       12
Studente_07           8        0
Studente_08          14       11
Studente_09          10        6
Studente_10           8       19

Statistiche descrittive per colonna
------------------------------------------------------------
       segreteria    sistema
count   10.000000  10.000000
mean    10.200000   8.800000
std      5.159673   5.711587
min      3.000000   0.000000
25%      7.250000   6.000000
50%      9.000000   7.000000
75%     14.000000  11.750000
max     19.000000  19.000000


## 4️⃣ 5️⃣ Task 4 e 5 — Riconciliazione delle due fonti

Veniamo al cuore del notebook: confrontare le due fonti dato per dato per rispondere a due domande operative complementari:

- **Task 4** — *quali studenti sono tracciati da entrambe le fonti?* (intersezione segreteria ∩ sistema)
- **Task 5** — *quali studenti la segreteria ha tracciato ma che il sistema non ha segnalato?* (differenza segreteria \ sistema)

Lo strumento è il metodo `isin()` di Pandas, che restituisce una maschera booleana: `True` per gli elementi presenti nell'altra serie, `False` per quelli assenti. Applicata alla serie originale, la maschera filtra gli elementi comuni; con l'operatore di negazione `~` si ottiene il complemento (elementi non comuni).

Una nota sulla **gestione dei duplicati**: se un valore appare più volte nella serie filtrata, l'output lo mostra più volte. È il comportamento corretto — stiamo filtrando una Series mantenendo i suoi indici originali, non costruendo un insieme matematico. Per ottenere i soli valori distinti basterebbe applicare `.unique()` al risultato.

Il Task 5 è **asimmetrico**: chiede *segreteria \ sistema*, non la simmetrica *sistema \ segreteria*. Operativamente, è il caso più interessante per la dirigenza: gli studenti che la segreteria ha colto manualmente e che il sistema non ha rilevato sono potenziali **falsi negativi del sistema informatico**, casi che varrebbe la pena indagare.

In [4]:
def elementi_comuni(serie_A, serie_B):
    """
    Trova gli elementi comuni tra due pandas Series.

    Args:
        serie_A (pandas.Series): Prima serie di elementi.
        serie_B (pandas.Series): Seconda serie di elementi.

    Returns:
        pandas.Series: Serie contenente solo gli elementi di serie_A
                       presenti anche in serie_B, con gli indici originali.
    """
    # Maschera booleana via isin: True per gli elementi di serie_A
    # presenti in serie_B. Filtriamo serie_A mantenendo i suoi indici.
    comuni = serie_A[serie_A.isin(serie_B)]

    # Stampa formattata
    print("Studenti tracciati da entrambe le fonti — intersezione")
    print("-" * 60)
    print(comuni)
    print(f"\nNumero di elementi comuni: {len(comuni)}")
    print(f"Valori distinti comuni:    {sorted(comuni.unique().tolist())}")

    return comuni


# Applicazione: intersezione segreteria ∩ sistema
studenti_in_entrambe = elementi_comuni(serie_segreteria, serie_sistema)

Studenti tracciati da entrambe le fonti — intersezione
------------------------------------------------------------
1     4
3    19
4     7
Name: segreteria, dtype: int64

Numero di elementi comuni: 3
Valori distinti comuni:    [4, 7, 19]


In [5]:
def elementi_non_comuni(serie_A, serie_B):
    """
    Trova gli elementi di serie_A che non sono presenti in serie_B.

    Args:
        serie_A (pandas.Series): Prima serie di elementi.
        serie_B (pandas.Series): Seconda serie di elementi.

    Returns:
        pandas.Series: Serie contenente solo gli elementi di serie_A
                       NON presenti in serie_B, con gli indici originali.
    """
    # L'operatore ~ inverte la maschera booleana di isin:
    # True diventa False e viceversa. Filtriamo così gli elementi esclusivi.
    non_comuni = serie_A[~serie_A.isin(serie_B)]

    # Stampa formattata
    print("Studenti tracciati solo dalla segreteria — differenza")
    print("-" * 60)
    print(non_comuni)
    print(f"\nNumero di elementi solo in segreteria: {len(non_comuni)}")
    print(f"Valori distinti solo in segreteria:    {sorted(non_comuni.unique().tolist())}")

    return non_comuni


# Applicazione: differenza segreteria \ sistema
studenti_solo_segreteria = elementi_non_comuni(serie_segreteria, serie_sistema)

Studenti tracciati solo dalla segreteria — differenza
------------------------------------------------------------
0    15
2     3
5    14
6     8
7    14
8    10
9     8
Name: segreteria, dtype: int64

Numero di elementi solo in segreteria: 7
Valori distinti solo in segreteria:    [3, 8, 10, 14, 15]


## 6️⃣ Task 6 — Reshape: da Series lineare a DataFrame tabellare

Un caso d'uso ricorrente in qualsiasi workflow di analisi dati è ricevere informazioni in **formato lineare** (un valore per riga, tutti in colonna) e doverle riorganizzare in **formato tabellare** (una griglia leggibile). Tipicamente succede quando un sistema esterno esporta i dati in formato "lungo" e noi ne abbiamo bisogno in formato "largo" per leggerli o per costruire un report.

Qui simuliamo questo scenario applicato al programma di tutoraggio: il sistema informatico restituisce una Series lineare di **35 voti** (compresi tra 1 e 9) — i voti di **7 studenti** del tutorato in **5 prove** svolte durante l'anno. Il nostro obiettivo è riorganizzarli in una tabella 7 × 5, dove ogni riga è uno studente e ogni colonna una prova.

La tecnica chiave è il metodo `.reshape(righe, colonne)` di NumPy, applicato all'array sottostante della Series (`series.values`). Il risultato — una matrice 2D — viene poi avvolto in un `pd.DataFrame` con etichette di colonna e di indice narrative.

In [6]:
# Series lineare di 35 voti (range 1-9): voti grezzi esportati dal sistema
# in formato "lungo", da riorganizzare in formato tabellare 7 × 5.
# Nota: np.random.randint(1, 10, ...) genera valori in [1, 10), cioè 1-9 inclusi.
series = pd.Series(np.random.randint(1, 10, 35))

# Reshape: dall'array 1D di 35 elementi a una matrice 2D di 7 righe × 5 colonne.
# Il metodo reshape opera sull'array sottostante (series.values).
matrice = series.values.reshape(7, 5)

# Costruzione del DataFrame con etichette narrative per colonne e indice
df_voti_prove = pd.DataFrame(
    matrice,
    columns=["Prova 1", "Prova 2", "Prova 3", "Prova 4", "Prova 5"]
)
df_voti_prove.index = [f"Studente {chr(65 + i)}" for i in range(7)]
df_voti_prove.index.name = "Studente"

# Visualizzazione
print("Voti per studente e prova — tutorato")
print("-" * 60)
print(df_voti_prove)

Voti per studente e prova — tutorato
------------------------------------------------------------
            Prova 1  Prova 2  Prova 3  Prova 4  Prova 5
Studente                                               
Studente A        6        9        9        8        6
Studente B        1        1        3        9        7
Studente C        5        8        4        4        9
Studente D        4        1        2        1        1
Studente E        7        8        8        4        1
Studente F        8        8        8        1        6
Studente G        5        4        2        4        2


## ➕ Extra — Il report finale per la dirigenza

> *Esercizio aggiuntivo, ideato e risolto durante il corso, qui rivisitato come deliverable conclusivo del progetto.*

Questo è il momento in cui il taglio tecnico cede il passo al **deliverable vero e proprio**. Mettiamo insieme tutto quello che abbiamo costruito (Series, DataFrame, indicizzazione personalizzata, statistiche descrittive) e produciamo il **report che la dirigenza userà operativamente** nei propri incontri settimanali: un quadro tabellare dei punteggi degli studenti del tutorato nelle quattro materie classiche, accompagnato dalle statistiche di sintesi per materia.

Strutturiamo il codice in due funzioni separate per chiarezza di responsabilità: `genera_dataframe_studenti(num_studenti)` costruisce il DataFrame con voti simulati, e `calcola_statistiche(df)` ne calcola media e deviazione standard per ogni colonna numerica. Lavoriamo su **15 studenti × 4 materie** (Latino, Greco, Italiano, Storia), con voti scolastici nel range 1-10. Il seed di riproducibilità è fissato all'interno della funzione di generazione, scelta che garantisce output stabili indipendentemente dal contesto della sessione.

In [7]:
def genera_dataframe_studenti(num_studenti: int = 15) -> pd.DataFrame:
    """
    Genera un DataFrame contenente voti simulati di studenti del tutorato
    nelle quattro materie classiche del Liceo.

    Args:
        num_studenti (int): Numero di studenti da generare. Default: 15.

    Returns:
        pd.DataFrame: DataFrame con voti simulati nel range 1-10,
                      indicizzato per studente.
    """
    # Seed locale per garantire riproducibilità della funzione
    # indipendentemente dallo stato del generatore casuale del notebook.
    np.random.seed(42)

    dati = {
        "Latino": np.random.randint(1, 11, num_studenti),
        "Greco": np.random.randint(1, 11, num_studenti),
        "Italiano": np.random.randint(1, 11, num_studenti),
        "Storia": np.random.randint(1, 11, num_studenti),
    }

    df = pd.DataFrame(dati)
    df.index = [f"Studente_{i+1:02d}" for i in range(num_studenti)]
    df.index.name = "Studente"

    return df


def calcola_statistiche(df: pd.DataFrame) -> tuple:
    """
    Calcola media e deviazione standard per ciascuna colonna numerica.

    Args:
        df (pd.DataFrame): DataFrame con almeno una colonna numerica.

    Returns:
        tuple: (medie, deviazioni_standard) come pandas Series, una
               entry per ogni colonna numerica del DataFrame.
    """
    colonne_numeriche = df.select_dtypes(include=np.number)
    medie = colonne_numeriche.mean()
    deviazioni_standard = colonne_numeriche.std()

    return medie, deviazioni_standard


# Generazione del DataFrame e calcolo delle statistiche
df_report = genera_dataframe_studenti(15)
medie, deviazioni = calcola_statistiche(df_report)

# Stampa del report finale
print("Report finale — punteggi degli studenti del tutorato")
print("-" * 60)
print(df_report)

print()
print("Media per materia")
print("-" * 60)
print(medie.round(2))

print()
print("Deviazione standard per materia")
print("-" * 60)
print(deviazioni.round(2))

Report finale — punteggi degli studenti del tutorato
------------------------------------------------------------
             Latino  Greco  Italiano  Storia
Studente                                    
Studente_01       7      5         9       5
Studente_02       4      2         3       2
Studente_03       8      8         5       4
Studente_04       5      6         3       7
Studente_05       7      2         7       8
Studente_06      10      5         5       3
Studente_07       3      1         9       1
Studente_08       7     10         7       4
Studente_09       8      6         2       2
Studente_10       5      9         4       8
Studente_11       4      1         9       4
Studente_12       8     10         2       2
Studente_13       8      3        10       6
Studente_14       3      7         9       6
Studente_15       6      4        10      10

Media per materia
------------------------------------------------------------
Latino      6.20
Greco       5.27
Italian

## 📋 Sintesi del Notebook 3

Il Notebook 3 ha portato i dati del Liceo nel mondo di **Pandas**, dove tabelle etichettate, indici nominati e statistiche descrittive sono cittadini di prima classe. I sei task canonici hanno mostrato i mattoni fondamentali — costruzione di Series da fonti diverse (lista vs array), filtri modulari, costruzione di DataFrame, riconciliazione tra fonti tramite `isin()`, reshape strutturale — e l'Extra li ha messi insieme nel **report finale** consegnabile alla dirigenza.

Il dato narrativamente più interessante emerso è la **riconciliazione tra segreteria e sistema informatico**: 7 occorrenze (5 studenti distinti) tracciati manualmente dalla segreteria che il sistema non ha rilevato — potenziali falsi negativi che meriterebbero un follow-up. Il report finale mostra **Storia (media 4.80)** come materia più critica e **Greco (deviazione standard 3.10)** come materia con la maggiore eterogeneità di rendimento.

---

## 🎯 Chiusura del progetto Liceo Virgilio

Il progetto si chiude con **tre deliverable distinti** consegnati alla dirigenza:

- **📘 Notebook 1** — La fotografia diagnostica della 3A: chi è in difficoltà, quanti studenti rischiano la promozione, quale tasso di promozione complessivo
- **📗 Notebook 2** — La visione multidimensionale dell'intero terzo anno: confronto tra sezioni e annualità, rendimento aggregato, traiettorie temporali normalizzate
- **📕 Notebook 3** — Il report tabellare operativo: riconciliazione delle fonti dati e quadro statistico per materia, in formato leggibile e riutilizzabile

I tre notebook insieme rispondono alla richiesta iniziale della dirigenza — **sostituire le valutazioni aneddotiche con evidenze quantitative** — coprendo tre livelli di analisi (locale, multidimensionale, sintetico) con tre strumenti (NumPy nativo, NumPy multidimensionale, Pandas tabellare).

---

## 🔮 Possibili sviluppi futuri

Il progetto si presta a estensioni naturali in tre direzioni:

- **Visualizzazioni** — istogrammi, heatmap e grafici di trend con `matplotlib` o `seaborn`
- **Integrazione con dati reali** — sostituire i dati simulati con export effettivi del sistema informatico
- **Automazione del report** — generazione periodica con esportazione diretta in Excel o PDF

Sono direzioni che lasciamo aperte: il progetto ha consegnato la **base analitica solida** sulla quale eventuali estensioni future potranno innestarsi.